C26. Перенос механизма на MedMCQA и PubMedQA.....


In [1]:
import sys, os, re
from itertools import combinations

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/benchmark_transfer')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools

def letter(s):
    s = str(s).strip().upper()
    m = re.match('^[\\(\\[]?([ABCD])', s)
    return m.group(1) if m else None

def load_med(rel):
    d = rd_any(rel, 'case_id')
    if d is None:
        return None
    truth = d['correct_option_letter'].map(letter)
    if 'baseline_selected_option' in d.columns:
        pred = d['baseline_selected_option'].map(letter)
        kind = 'одиночная'
    elif 'delphi_consensus_selected_option' in d.columns:
        pred = d['delphi_consensus_selected_option'].map(letter)
        if pred.notna().mean() < 0.5 and 'winner_selected_option' in d.columns:
            pred = d['winner_selected_option'].map(letter)
        kind = 'панель'
    else:
        return None
    return (d, truth, pred, kind)
ARMS = {}
for name, rel in MEDMCQA.items():
    r = load_med(rel)
    if r is None:
        print('пропуск:', name)
        continue
    ARMS[name] = r
print('плеч MedMCQA загружено:', len(ARMS))


плеч MedMCQA загружено: 9


In [2]:
rows = []
for name, (d, t, p, kind) in ARMS.items():
    ok = t.notna() & p.notna()
    n = int(ok.sum())
    acc = float((p[ok] == t[ok]).mean())
    lo, hi = ci_wilson(acc, n)
    rows.append({'плечо': name, 'тип': kind, 'n': n, 'accuracy': acc, 'CI низ': lo, 'CI верх': hi, 'покрытие ответов': float(p.notna().mean())})
ACC = pd.DataFrame(rows).set_index('плечо').sort_values('accuracy', ascending=False)
show(ACC, 'MedMCQA: точность по плечам (интервалы Уилсона)')



MedMCQA: точность по плечам (интервалы Уилсона)
------------------------------------------------------------
                             тип     n  accuracy  CI низ  CI верх  покрытие ответов
плечо                                                                              
P-deepseektrio            панель  1000    0.8930  0.8723   0.9107            1.0000
baseline deepseekv4    одиночная   995    0.8834  0.8620   0.9019            0.9950
P-ac (gpt/qwen/ds)        панель  1000    0.8810  0.8595   0.8996            1.0000
P-base-like               панель  1000    0.8760  0.8541   0.8950            1.0000
baseline qwen36        одиночная   980    0.8673  0.8447   0.8872            0.9800
P-ac (gpt/qwen/alice)     панель  1000    0.8620  0.8392   0.8820            1.0000
baseline alice         одиночная   991    0.8234  0.7984   0.8459            0.9910
baseline qwen          одиночная   989    0.8231  0.7980   0.8456            0.9890
baseline gpt           одиночная   997    0.8154  

In [3]:
PANEL = 'P-ac (gpt/qwen/ds)'
MEMB = ['baseline gpt', 'baseline qwen', 'baseline deepseekv4']
if PANEL in ARMS and all((m in ARMS for m in MEMB)):
    dp, tp, pp, _ = ARMS[PANEL]
    corr = {}
    for m in MEMB:
        dm, tm, pm, _ = ARMS[m]
        corr[m] = (pm == tm).astype(float).where(pm.notna() & tm.notna())
    C = pd.DataFrame(corr).dropna()
    panel_ok = (pp == tp).astype(float).where(pp.notna() & tp.notna())
    ix = C.index.intersection(panel_ok.dropna().index)
    C, po = (C.loc[ix], panel_ok.loc[ix])
    mean_m, max_m, maj_m = (C.mean(axis=1), C.max(axis=1), (C.sum(axis=1) >= 2).astype(float))
    rows = []
    for lab, ref in [('среднее участников', mean_m), ('покейсный максимум', max_m), ('большинство голосов', maj_m)]:
        d_ = po - ref
        rows.append({'контраст': 'панель − %s' % lab, 'n': len(d_), 'панель': float(po.mean()), 'референс': float(ref.mean()), 'Δ': float(d_.mean()), 'SE': se_clt(d_), 'p': float(stats.wilcoxon(d_)[1]) if (d_ != 0).any() else np.nan})
    show(pd.DataFrame(rows).set_index('контраст'), 'MedMCQA: эмерджентность панели (n = %d)' % len(ix))
    print()
    print('НА ОСНОВНОЙ ЗАДАЧЕ (из A1a): панель превосходила покейсный максимум')
    print('на +0.088 (p < 1e-16). Сравните со значением выше.')



MedMCQA: эмерджентность панели (n = 982)
------------------------------------------------------------
                                n  панель  референс       Δ     SE      p
контраст                                                                 
панель − среднее участников   982  0.8849    0.8435  0.0414 0.0067 0.0000
панель − покейсный максимум   982  0.8849    0.9389 -0.0540 0.0079 0.0000
панель − большинство голосов  982  0.8849    0.8707  0.0143 0.0072 0.0477

НА ОСНОВНОЙ ЗАДАЧЕ (из A1a): панель превосходила покейсный максимум
на +0.088 (p < 1e-16). Сравните со значением выше.


In [4]:
if 'C' in dir():
    CM = C.corr()
    show(CM, 'MedMCQA: корреляции правильности между моделями')
    rs = [CM.iloc[i, j] for i, j in combinations(range(CM.shape[0]), 2)]
    rbar = float(np.mean(rs))
    k = CM.shape[0]
    keff = k / (1 + (k - 1) * rbar)
    print()
    print('MedMCQA:  ρ̄ = %.3f   k_eff = %.3f   предел 1/ρ̄ = %.2f' % (rbar, keff, 1 / rbar))
    print('llmfmed (из B9): ρ̄ = 0.581   k_eff = 1.387   предел 1.72')
    print()
    print('Близкие значения означают, что информационный потолок пула моделей —')
    print('свойство самих моделей, а не конкретной задачи.')



MedMCQA: корреляции правильности между моделями
------------------------------------------------------------
                     baseline gpt  baseline qwen  baseline deepseekv4
baseline gpt               1.0000         0.4091               0.4832
baseline qwen              0.4091         1.0000               0.5039
baseline deepseekv4        0.4832         0.5039               1.0000

MedMCQA:  ρ̄ = 0.465   k_eff = 1.554   предел 1/ρ̄ = 2.15
llmfmed (из B9): ρ̄ = 0.581   k_eff = 1.387   предел 1.72

Близкие значения означают, что информационный потолок пула моделей —
свойство самих моделей, а не конкретной задачи.


In [5]:
rows = []
for pname, comp in [('P-ac (gpt/qwen/ds)', MEMB)]:
    pass
POOL = {m.replace('baseline ', ''): ARMS[m] for m in ARMS if m.startswith('baseline ')}
V = {}
for nm, (d, t, p, _) in POOL.items():
    V[nm] = (p == t).astype(float).where(p.notna() & t.notna())
VV = pd.DataFrame(V).dropna()
CM_all = VV.corr()
show(CM_all, 'Корреляции правильности между всеми одиночными моделями MedMCQA')
print('\nсредняя попарная корреляция по пулу: %.3f' % np.mean([CM_all.iloc[i, j] for i, j in combinations(range(CM_all.shape[0]), 2)]))



Корреляции правильности между всеми одиночными моделями MedMCQA
---------------------------------------------------------------
            deepseekv4    gpt   qwen  alice  qwen36
deepseekv4      1.0000 0.4829 0.5045 0.5222  0.6072
gpt             0.4829 1.0000 0.4028 0.5048  0.5825
qwen            0.5045 0.4028 1.0000 0.6600  0.4551
alice           0.5222 0.5048 0.6600 1.0000  0.5471
qwen36          0.6072 0.5825 0.4551 0.5471  1.0000

средняя попарная корреляция по пулу: 0.527


In [6]:
if PANEL in ARMS:
    dp = ARMS[PANEL][0]
    tp = ARMS[PANEL][1]
    ps = sorted(set((c.split('__', 1)[1] for c in dp.columns if c.startswith('r1_selected_option__'))))
    print('участники панели:', ps)
    lab = {}
    for r in (1, 2, 3):
        for p in ps:
            c = 'r%d_selected_option__%s' % (r, p)
            lab[r, p] = dp[c].map(letter) if c in dp.columns else pd.Series(index=dp.index, dtype=object)
    R1 = pd.DataFrame({p: lab[1, p] for p in ps})
    modal = R1.mode(axis=1)
    modal = modal[0] if len(modal.columns) else pd.Series(index=dp.index, dtype=object)
    rr = pd.to_numeric(dp.get('delphi_rounds_executed'), errors='coerce')
    rows = []
    for p in ps:
        l1 = lab[1, p]
        last = lab[3, p].where(lab[3, p].notna(), lab[2, p])
        last = last.where(last.notna(), l1)
        rows.append(pd.DataFrame({'pid': p, 'case': dp.index, 'r1': l1.to_numpy(), 'last': last.to_numpy(), 'truth': tp.to_numpy(), 'modal': modal.to_numpy(), 'rounds': rr.to_numpy()}))
    L = pd.concat(rows, ignore_index=True).dropna(subset=['r1', 'truth'])
    L['r1_ok'] = (L['r1'] == L['truth']).astype(int)
    L['last_ok'] = (L['last'] == L['truth']).astype(int)
    L['minority'] = (L['r1'] != L['modal']).astype(int)
    E = L[L['rounds'] > 1]
    sp = int(((E['r1_ok'] == 1) & (E['last_ok'] == 0)).sum())
    rp = int(((E['r1_ok'] == 0) & (E['last_ok'] == 1)).sum())
    print()
    print('наблюдений (вопрос × участник) с раундами > 1: %d' % len(E))
    print('порча %d, починка %d, нетто %+d' % (sp, rp, rp - sp))
    if sp + rp:
        print('точный биномиальный тест: p = %s' % fmt_p(stats.binomtest(rp, sp + rp, 0.5).pvalue))
    print('llmfmed (из B6): порча 111, починка 194, p = 2.3e-06')
    sub = E[E['r1_ok'] == 1]
    cnt = pd.crosstab(sub['minority'], (sub['r1_ok'] == 1) & (sub['last_ok'] == 0))
    if cnt.shape == (2, 2):
        tab = pd.DataFrame({'n': cnt.sum(axis=1), 'испорчено': cnt.iloc[:, 1], 'доля порчи': cnt.iloc[:, 1] / cnt.sum(axis=1)})
        tab.index = ['в большинстве R1', 'в меньшинстве R1']
        show(tab, 'MedMCQA: порча среди изначально правых')
        orat = cnt.iloc[1, 1] * cnt.iloc[0, 0] / max(cnt.iloc[1, 0] * cnt.iloc[0, 1], 1e-09)
        print('отношение шансов (меньшинство/большинство) = %.2f, тест Фишера p = %s' % (orat, fmt_p(stats.fisher_exact(cnt.to_numpy())[1])))
        print('llmfmed (из B6): OR = 46.6, p = 1.5e-54')


участники панели: ['deepseekv4_default', 'gpt120b_default', 'qwen_default']



наблюдений (вопрос × участник) с раундами > 1: 477
порча 34, починка 119, нетто +85
точный биномиальный тест: p = 3.0e-12
llmfmed (из B6): порча 111, починка 194, p = 2.3e-06

MedMCQA: порча среди изначально правых
------------------------------------------------------------
                    n  испорчено  доля порчи
в большинстве R1  199         14      0.0704
в меньшинстве R1   41         20      0.4878
отношение шансов (меньшинство/большинство) = 12.59, тест Фишера p = 1.3e-09
llmfmed (из B6): OR = 46.6, p = 1.5e-54


In [7]:
if PANEL in ARMS:
    rows = []
    for r_ in sorted(rr.dropna().unique()):
        sel = rr == r_
        pp_ = ARMS[PANEL][2]
        okk = sel & tp.notna() & pp_.notna()
        rows.append({'раундов': int(r_), 'вопросов': int(okk.sum()), 'accuracy панели': float((pp_[okk] == tp[okk]).mean())})
    show(pd.DataFrame(rows).set_index('раундов'), 'Точность в зависимости от числа исполненных раундов')
    print()
    print('ВНИМАНИЕ: это НЕ эффект раундов. Число раундов определяется тем,')
    print('согласились ли участники сразу, а согласие сильнее на лёгких')
    print('вопросах. Здесь смешаны трудность и длительность процедуры.')
    print('Корректная оценка эффекта раундов требует стадийных прогонов,')
    print('которых для MedMCQA нет.')



Точность в зависимости от числа исполненных раундов
------------------------------------------------------------
         вопросов  accuracy панели
раундов                           
1             827           0.9274
2             108           0.6759
3              65           0.6308

ВНИМАНИЕ: это НЕ эффект раундов. Число раундов определяется тем,
согласились ли участники сразу, а согласие сильнее на лёгких
вопросах. Здесь смешаны трудность и длительность процедуры.
Корректная оценка эффекта раундов требует стадийных прогонов,
которых для MedMCQA нет.


In [8]:
def safe(v, default=np.nan):
    try:
        return float(v)
    except Exception:
        return default
rows = [{'механизм': 'k_eff при k=3', 'llmfmed': 1.387, 'MedMCQA': safe(keff) if 'keff' in dir() else np.nan}, {'механизм': 'ρ̄ корреляции ошибок', 'llmfmed': 0.581, 'MedMCQA': safe(rbar) if 'rbar' in dir() else np.nan}, {'механизм': 'превосходство над покейсным максимумом', 'llmfmed': 0.088, 'MedMCQA': np.nan}, {'механизм': 'нетто починок в раундах', 'llmfmed': 83.0, 'MedMCQA': rp - sp if 'rp' in dir() else np.nan}]
if PANEL in ARMS and 'max_m' in dir():
    rows[2]['MedMCQA'] = float((po - max_m).mean())
T = pd.DataFrame(rows).set_index('механизм')
T['перенеслось'] = np.where(T['MedMCQA'].isna(), 'нет данных', np.where(np.sign(T['MedMCQA']) == np.sign(T['llmfmed']), 'знак совпал', 'ЗНАК ПРОТИВОПОЛОЖЕН'))
show(T, 'Перенос механизмов: llmfmed против MedMCQA')



Перенос механизмов: llmfmed против MedMCQA
------------------------------------------------------------
                                        llmfmed  MedMCQA          перенеслось
механизм                                                                     
k_eff при k=3                            1.3870   1.5537          знак совпал
ρ̄ корреляции ошибок                     0.5810   0.4654          знак совпал
превосходство над покейсным максимумом   0.0880  -0.0540  ЗНАК ПРОТИВОПОЛОЖЕН
нетто починок в раундах                 83.0000  85.0000          знак совпал


In [9]:
save(ACC.reset_index(), 'medmcqa_pubmedqa_mechanism_transfer_medmcqa_accuracy.csv')
save(CM_all.reset_index(), 'medmcqa_pubmedqa_mechanism_transfer_medmcqa_corr.csv')
save(T.reset_index(), 'medmcqa_pubmedqa_mechanism_transfer_transfer.csv')


сохранено: out/medmcqa_pubmedqa_mechanism_transfer_medmcqa_accuracy.csv  (9 строк)
сохранено: out/medmcqa_pubmedqa_mechanism_transfer_medmcqa_corr.csv  (5 строк)
сохранено: out/medmcqa_pubmedqa_mechanism_transfer_transfer.csv  (4 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\benchmark_transfer\\results\\medmcqa_pubmedqa_mechanism_transfer_transfer.csv'